# GPU Dev SDK — Quickstart

Interactive notebook to test the SDK. Run cells in order.

In [ ]:
# Install SDK (run once)
!pip install -e .. -q

In [ ]:
from gpu_dev import GpuDev, GpuDevConfig

client = GpuDev()
print(f"SDK loaded, version {__import__('gpu_dev').__version__}")

## 1. Check GPU Availability

In [ ]:
avail = client.availability()

print(f"{'GPU Type':15s} {'Available':>10s} {'Total':>8s} {'Max Reservable':>15s}")
print("-" * 50)
for gpu_type, info in sorted(avail.items()):
    if info.total > 0:
        print(f"{gpu_type:15s} {info.available:>10d} {info.total:>8d} {info.max_reservable:>15d}")

## 2. List Existing Reservations

In [ ]:
reservations = client.list()

if not reservations:
    print("No active reservations")
else:
    for sb in reservations:
        print(f"{sb.id[:8]}  {sb.gpu_count}x {sb.gpu_type:10s}  {sb.status.value:10s}  {sb.name or ''}")

## 3. Reserve a T4 GPU

In [ ]:
sandbox = client.reserve(
    gpu_type="t4",
    gpu_count=1,
    hours=0.5,
    name="sdk-test",
)

print(f"Reservation: {sandbox.id[:8]}")
print(f"Status:      {sandbox.status.value}")
print(f"SSH:         {sandbox.ssh_command}")
print(f"Pod:         {sandbox.pod_name}")
print(f"Expires:     {sandbox.expires_at}")

## 4. Execute Commands

In [ ]:
# Check GPU
result = sandbox.exec("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(f"GPU: {result.stdout.strip()}")
print(f"Exit code: {result.exit_code}")

In [ ]:
# Check CUDA
result = sandbox.exec("python3 -c 'import torch; print(f\"PyTorch {torch.__version__}, CUDA {torch.cuda.get_device_name(0)}\")' 2>&1")
print(result.stdout.strip())

In [ ]:
# Run a quick benchmark
result = sandbox.exec("""
python3 -c '
import torch, time
x = torch.randn(4096, 4096, device="cuda")
torch.cuda.synchronize()
t0 = time.time()
for _ in range(100):
    y = x @ x
torch.cuda.synchronize()
print(f"matmul 4096x4096 x100: {(time.time()-t0)*1000:.0f}ms")
'
""", timeout=30)
print(result.stdout.strip())

## 5. File Transfer

In [ ]:
# Write a local file
with open("/tmp/sdk_test.py", "w") as f:
    f.write('import torch\nprint(f"CUDA available: {torch.cuda.is_available()}")\n')

# Upload and run
sandbox.upload("/tmp/sdk_test.py", "/home/dev/sdk_test.py")
result = sandbox.exec("python3 /home/dev/sdk_test.py")
print(result.stdout.strip())

In [ ]:
# Download a file
sandbox.exec("echo 'results from GPU' > /home/dev/output.txt")
sandbox.download("/home/dev/output.txt", "/tmp/sdk_output.txt")
print(open("/tmp/sdk_output.txt").read())

## 6. Disk Persistence

In [ ]:
# Write a marker file to the persistent disk
sandbox.exec("echo 'sdk_persistence_test' > /home/dev/persist_check.txt")
result = sandbox.exec("cat /home/dev/persist_check.txt")
print(f"Written: {result.stdout.strip()}")

In [ ]:
# List disks
for disk in client.disks():
    status = "IN USE" if disk.in_use else "available"
    print(f"  {disk.name:20s}  {disk.size_gb:>4d}GB  {disk.snapshot_count} snapshots  {status}")

## 7. Cancel Reservation

In [ ]:
sandbox.cancel()
print(f"Cancelled: {sandbox.id[:8]} (status: {sandbox.status.value})")

## 8. Spot Instance

Spot instances are cheaper but may be preempted. Availability depends on AWS capacity.

In [ ]:
try:
    spot_sb = client.reserve(
        gpu_type="t4",
        gpu_count=1,
        hours=0.5,
        spot=True,
        name="sdk-spot-test",
        timeout_minutes=5,
    )
    print(f"Spot reservation: {spot_sb.id[:8]}")
    print(f"Status: {spot_sb.status.value}")
    result = spot_sb.exec("nvidia-smi -L")
    print(result.stdout.strip())
    spot_sb.cancel()
    print("Spot cancelled")
except Exception as e:
    print(f"Spot not available: {e}")

## 9. Context Manager Pattern

Auto-cancels when the block exits — prevents leaked reservations.

In [ ]:
with client.reserve(gpu_type="t4", hours=0.25, name="sdk-ctx-test") as sb:
    result = sb.exec("hostname && nvidia-smi -L")
    print(result.stdout.strip())

print(f"Auto-cancelled: {sb.status.value}")

## 10. Reconnect to Existing Reservation

In [ ]:
# List active reservations and reconnect
active = client.list(status=["active"])
if active:
    existing = client.get(active[0].id)
    print(f"Reconnected to {existing.id[:8]} ({existing.gpu_count}x {existing.gpu_type})")
    result = existing.exec("uptime")
    print(result.stdout.strip())
else:
    print("No active reservations to reconnect to")

## 11. Error Handling

In [ ]:
from gpu_dev import GpuDevValidationError, GpuDevNotFoundError

# Invalid GPU type
try:
    client.reserve(gpu_type="nvidia-rtx-9090")
except GpuDevValidationError as e:
    print(f"Validation error: {e}")

# Too many GPUs
try:
    client.reserve(gpu_type="t4", gpu_count=16)
except GpuDevValidationError as e:
    print(f"Validation error: {e}")

# Non-existent reservation
try:
    client.get("nonexistent-id-that-doesnt-exist")
except GpuDevNotFoundError as e:
    print(f"Not found: {e}")